# Model Training — Logistic Regression 

In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

SEED = 42

In [2]:
X_train = pd.read_csv("X_train.xls")
y_train = pd.read_csv("y_train.xls")["Outcome"]

# prepare data

In [3]:
zero_columns = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
X = X_train.copy()
X[zero_columns] = X[zero_columns].replace(0, np.nan)

imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()
X = scaler.fit_transform(imputer.fit_transform(X))

# train and save

In [4]:
model = LogisticRegression(C=1.0, solver="lbfgs", max_iter=1000, random_state=SEED)
model.fit(X, y_train)

joblib.dump(model, "logistic_regression_model.joblib")
joblib.dump(imputer, "imputer.joblib")
joblib.dump(scaler, "scaler.joblib")

['scaler.joblib']

# evaluate on test data

In [5]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

X_test = pd.read_csv("X_test.xls")
y_test = pd.read_csv("y_test.xls")["Outcome"]

# same preparation as training data
X_t = X_test.copy()
X_t[zero_columns] = X_t[zero_columns].replace(0, np.nan)
X_t = scaler.transform(imputer.transform(X_t))

y_pred = model.predict(X_t)
y_prob = model.predict_proba(X_t)[:, 1]

print("Accuracy :", round(accuracy_score(y_test, y_pred), 3))
print("Precision:", round(precision_score(y_test, y_pred), 3))
print("Recall   :", round(recall_score(y_test, y_pred), 3))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy : 0.708
Precision: 0.6
Recall   : 0.5
Confusion matrix:
[[82 18]
 [27 27]]


# save results

In [6]:
results = X_test.copy()
results["actual"] = y_test
results["predicted"] = y_pred
results["probability"] = y_prob.round(4)

results.to_csv("prediction_results.csv", index=False)